# tmol-mutscan on Google Colab

**Interface ΔΔG mutational scanning on a free GPU.**

This notebook runs [`tmol-mutscan`](https://github.com/pablo-arantes/tmol-mutscan)
on Google Colab, so you can do ensemble-based saturation mutagenesis of a
binding interface without a local GPU or a cluster.

It relaxes your complex into a conformational ensemble, mutates the positions
you choose to every amino acid you ask for, and reports the predicted change in
binding energy (ΔΔG) for each substitution with full ensemble statistics —
standard deviation, confidence intervals, FDR-corrected significance and
convergence — plus publication-ready figures.

Works on **peptide–protein**, **protein–protein** and **protein–ligand**
complexes.

---

### Before you start: turn on the GPU

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

The notebook runs on CPU too, just much more slowly. Cell 1 checks what you
have and tells you where you stand.

### How to use this notebook

Run the cells in order, top to bottom (`Shift+Enter`, or Runtime → Run all).
Cells 1–4 set up the environment, cells 5–7 load and configure your structure,
cell 8 runs the scan, and cells 9–11 show and download the results.

The grey boxes on the right of some cells are input forms — change the values
there, no code editing needed.

### What you need

A single structure file containing **both binding partners**:

- `.pdb` — fine for protein-only complexes
- `.cif` / `.mmcif` / `.sdf` / `.mol2` — **use one of these if your complex
  contains a ligand**, because PDB files carry no bond records and ligand
  chemistry cannot be typed without them

You can fetch a structure straight from the PDB by ID, upload your own, or read
one from Google Drive.

### A word on runtime

Colab assigns whatever GPU is free (usually T4, sometimes L4 or A100), so speed
varies between sessions. Rather than guess, this notebook **measures its own
throughput** and prints poses/second and an ETA while the scan runs. Start with
the quick demo preset to calibrate on the GPU you were given, then size your
real run from that.

Free Colab sessions disconnect after about 90 minutes idle and 12 hours total,
and a GPU is not guaranteed. For long scans, use the *Save to Google Drive*
option in cell 11 so results survive a disconnect.

In [ ]:
#@title 1. Check the runtime { display-mode: "form" }
#@markdown Run this first. It reports your GPU and Python version **without**
#@markdown importing PyTorch, so it can tell you in advance whether the install
#@markdown will need a session restart.

import subprocess, sys, shutil
from importlib.metadata import PackageNotFoundError, version

print("=" * 72)
print("RUNTIME CHECK")
print("=" * 72)

# --- GPU -----------------------------------------------------------------
HAS_GPU = False
if shutil.which("nvidia-smi"):
    query = "--query-gpu=name,memory.total,driver_version --format=csv,noheader"
    probe = subprocess.run(
        ["nvidia-smi", *query.split()], capture_output=True, text=True
    )
    if probe.returncode == 0 and probe.stdout.strip():
        name, memory, driver = (f.strip() for f in probe.stdout.strip().split(","))
        print(f"GPU            : {name}")
        print(f"GPU memory     : {memory}")
        print(f"NVIDIA driver  : {driver}")
        HAS_GPU = True

if not HAS_GPU:
    print("GPU            : NONE DETECTED")

# --- Python --------------------------------------------------------------
major, minor = sys.version_info[:2]
print(f"Python         : {major}.{minor}.{sys.version_info[2]}")
PYTHON_OK = (major, minor) >= (3, 11)

# --- torch, without importing it ----------------------------------------
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = None
print(f"PyTorch        : {torch_version or 'not installed'}")

def _as_tuple(text):
    parts = []
    for piece in text.split("+")[0].split("."):
        try:
            parts.append(int(piece))
        except ValueError:
            break
    return tuple(parts)

TORCH_NEEDS_UPGRADE = torch_version is None or _as_tuple(torch_version) < (2, 11)

# --- verdict -------------------------------------------------------------
print("-" * 72)
if not PYTHON_OK:
    print("STOP: tmol needs Python 3.11 or newer, and this runtime is older.")
    print("      Colab's Python version is not user-selectable, so wait for")
    print("      Colab to update, or run tmol-mutscan locally instead.")
elif not HAS_GPU:
    print("NO GPU. The scan will still run correctly, but on the CPU, which is")
    print("much slower. To fix:")
    print("      Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    print("      -> Save, then re-run this cell.")
    print()
    print("If no GPU is available to you right now, carry on: use the")
    print("'quick_demo' preset in cell 7 and keep the pose count small.")
else:
    print("GPU and Python version are both fine.")

if TORCH_NEEDS_UPGRADE:
    print()
    print("NOTE: tmol needs PyTorch >= 2.11 and this runtime has "
          f"{torch_version or 'none'}.")
    print("      The install in cell 2 will upgrade it (a large download), and")
    print("      you will most likely have to restart the session afterwards.")
    print("      Cell 2 tells you exactly when.")
print("=" * 72)

## 2. Install

This installs `tmol` with its CUDA-12 operators and `tmol-mutscan` from GitHub,
in one `pip` call so the resolver sees them together.

Takes a few minutes — it pulls RDKit and Open Babel for ligand chemistry, and
possibly PyTorch.

**If cell 1 warned that PyTorch will be upgraded**, you will probably need to
do *Runtime → Restart session* after this cell, then continue from cell 3.
Do **not** re-run the install after restarting; the packages are already there.

In [ ]:
#@title 2. Install tmol and tmol-mutscan { display-mode: "form" }
#@markdown Which CUDA build of the TMol operators to install. `cu12` matches
#@markdown the GPUs Colab currently assigns; pick `cpu` only if you are
#@markdown deliberately running without a GPU.
cuda_build = "cu12"  #@param ["cu12", "cu13", "cpu"]
#@markdown Leave blank for the latest `main`, or give a tag/branch/commit to pin.
version_ref = ""  #@param {type:"string"}

import os, subprocess, sys, time

REPO = "https://github.com/pablo-arantes/tmol-mutscan.git"
ref = version_ref.strip() or "main"
extra = "" if cuda_build == "cpu" else f"[{cuda_build}]"
requirement = f"tmol-mutscan{extra} @ git+{REPO}@{ref}"

print(f"installing: {requirement}")
print("this takes a few minutes; output is summarised at the end\n")

started = time.perf_counter()
completed = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", requirement],
    capture_output=True,
    text=True,
)
elapsed = time.perf_counter() - started

tail = (completed.stdout + completed.stderr).strip().splitlines()
for line in tail[-15:]:
    print(line)

print(f"\npip finished in {elapsed / 60:.1f} min with exit code {completed.returncode}")

if completed.returncode != 0:
    print("\nINSTALL FAILED. The most common causes:")
    print("  - no matching wheel: check cell 1 said Python 3.11 or newer")
    print("  - a transient network error: just run this cell again")
    raise SystemExit("install failed; see the pip output above")

# Did pip change torch under us? If so this session's loaded torch is stale.
changed_torch = any(
    "torch" in line and ("Downloading" in line or "Installing" in line)
    for line in tail
)
torch_already_imported = "torch" in sys.modules

print("-" * 72)
if changed_torch or torch_already_imported:
    print("RESTART THE SESSION NOW:  Runtime -> Restart session")
    print("Then carry on from cell 3. Do not run this install cell again.")
else:
    print("Install complete, no restart needed. Continue to cell 3.")
print("-" * 72)

In [ ]:
#@title 3. Verify the installation { display-mode: "form" }
#@markdown This is the gate: it imports PyTorch and TMol and checks that
#@markdown **both** agree a GPU is usable. `tmol.has_cuda()` must be `True`,
#@markdown not just `torch.cuda.is_available()` — if only the latter is true,
#@markdown you have the CPU-only TMol wheel and the scan would silently crawl.

import torch, tmol, tmol_mutscan

print("=" * 72)
print(f"tmol-mutscan           : {tmol_mutscan.__version__}")
print(f"tmol                   : {tmol.__version__}")
print(f"torch                  : {torch.__version__}")
print(f"torch.cuda.is_available: {torch.cuda.is_available()}")
print(f"tmol.has_cuda          : {tmol.has_cuda()}")

DEVICE = "cpu"
if torch.cuda.is_available() and tmol.has_cuda():
    DEVICE = "cuda"
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU                    : {properties.name}")
    print(f"GPU memory             : {properties.total_memory / 1024**3:.1f} GiB")
elif torch.cuda.is_available() and not tmol.has_cuda():
    print("\nPyTorch sees a GPU but TMol has no CUDA operators installed.")
    print("Re-run cell 2 with cuda_build = 'cu12', then restart the session.")
    print("Falling back to the CPU for now.")
else:
    print("\nNo usable GPU; running on the CPU. Correct, but slow.")

print(f"\nDEVICE = {DEVICE!r}")
print("=" * 72)

# Smoke test: build a small pose from sequence and score it. This only
# establishes that the kernels load and return a finite number on this device;
# the pose has ideal extended geometry, so its score is expected to be large
# and positive, and its two chains do not touch.
import math, time

print("\nsmoke test: scoring a test pose on", DEVICE, "...")
from tmol.io import extended_pose_stack_from_sequences
from tmol.score import beta2016_score_function

device = torch.device(DEVICE)
pose = extended_pose_stack_from_sequences(["AAKLWGDEFTY"], device)
sfxn = beta2016_score_function(device)
scorer = sfxn.render_whole_pose_scoring_module(pose)

scorer(pose.coords)                      # warm up (kernel load / JIT)
if DEVICE == "cuda":
    torch.cuda.synchronize()
started = time.perf_counter()
total = float(scorer(pose.coords)[0])
if DEVICE == "cuda":
    torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - started) * 1000

if not math.isfinite(total):
    raise SystemExit(
        "the score is not finite - the installation is broken. Re-run cell 2."
    )
print(f"  scored an 11-residue pose in {elapsed_ms:.1f} ms "
      f"(total {total:.1f} REU, large because the geometry is ideal/extended)")
print("  OK - TMol works on", DEVICE)

## 4. Load your structure

Choose where the structure comes from. One file must contain **both** binding
partners.

For a **protein–ligand** complex, use mmCIF (or SDF/MOL2). A `.pdb` file has no
bond records, so the ligand cannot be typed from it — the notebook warns you if
it spots this.

In [ ]:
#@title 4. Get the structure { display-mode: "form" }
#@markdown Where to get the structure from.
input_source = "PDB ID"  #@param ["PDB ID", "Upload a file", "Google Drive"]

#@markdown **PDB ID** — a 4-character code, e.g. `1YCR` (p53 peptide + MDM2),
#@markdown `1A22` (growth hormone + receptor), `1BTY` (trypsin + benzamidine).
pdb_id = "1YCR"  #@param {type:"string"}
#@markdown Fetch as mmCIF instead of PDB. **Tick this if the complex contains a
#@markdown ligand** so that bond orders are preserved.
fetch_as_mmcif = False  #@param {type:"boolean"}

#@markdown **Google Drive** — path inside your Drive, e.g.
#@markdown `MyDrive/structures/complex.cif`. Drive is mounted on demand.
drive_path = ""  #@param {type:"string"}

import os, shutil
from pathlib import Path

WORKDIR = Path("/content/mutscan")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
STRUCTURE = None

if input_source == "PDB ID":
    import urllib.request, urllib.error

    code_ = pdb_id.strip().upper()
    if not code_:
        raise SystemExit("give a PDB ID, or switch input_source")
    suffix = "cif" if fetch_as_mmcif else "pdb"
    STRUCTURE = WORKDIR / f"{code_.lower()}.{suffix}"
    url = f"https://files.rcsb.org/download/{code_}.{suffix}"
    print(f"fetching {url}")
    try:
        urllib.request.urlretrieve(url, STRUCTURE)
    except urllib.error.HTTPError as problem:
        raise SystemExit(
            f"could not fetch {code_} as .{suffix} (HTTP {problem.code}). "
            "Check the ID, or try the other format."
        ) from None

elif input_source == "Upload a file":
    from google.colab import files

    print("choose a .pdb / .cif / .sdf / .mol2 file ...")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("nothing was uploaded")
    name = next(iter(uploaded))
    STRUCTURE = WORKDIR / name
    STRUCTURE.write_bytes(uploaded[name])

else:  # Google Drive
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    source = Path("/content/drive") / drive_path.strip().lstrip("/")
    if not source.is_file():
        raise SystemExit(f"not found in Drive: {source}")
    STRUCTURE = WORKDIR / source.name
    shutil.copy(source, STRUCTURE)

size_kb = STRUCTURE.stat().st_size / 1024
print(f"\nstructure: {STRUCTURE}  ({size_kb:.0f} kB)")

# Warn about the one mistake that silently degrades a ligand run.
if STRUCTURE.suffix.lower() == ".pdb":
    text = STRUCTURE.read_text(errors="ignore")
    het = {
        line[17:20].strip()
        for line in text.splitlines()
        if line.startswith("HETATM") and line[17:20].strip() not in {"HOH", "DOD"}
    }
    if het:
        print()
        print("WARNING: this PDB file contains HETATM residues: "
              f"{', '.join(sorted(het)[:10])}")
        print("PDB format carries no bond records, so ligand chemistry cannot be")
        print("typed from it. If any of those is a ligand you care about, re-run")
        print("this cell with fetch_as_mmcif ticked (or upload an mmCIF/SDF).")

## 5. Inspect the structure

This lists the chains, their residue numbering and what they contain. **Read the
output before configuring the scan** — the selections in cell 7 use exactly
these chain labels and residue numbers.

In [ ]:
#@title 5. Inspect chains and numbering { display-mode: "form" }
import subprocess, sys

# Invoked as a module rather than the `tmol-mutscan` script, so it works even
# if pip did not put the entry point on PATH.
result = subprocess.run(
    [sys.executable, "-m", "tmol_mutscan", "info",
     "-i", str(STRUCTURE), "--device", "cpu"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise SystemExit("could not read the structure; see the error above")

## 6. Selection syntax

`partner_a`, `partner_b` and `scan` all use the same notation. Terms join with
`+`.

| Term | Selects |
|---|---|
| `A` | every residue of chain A |
| `AB` or `A,B` | chains A and B |
| `A:17-29` | residues 17–29 of chain A (PDB numbering) |
| `A:5,9,14-18` | an explicit list and/or ranges |
| `resname:BEN` | every residue named `BEN` (ligands, ions) |
| `ligand` | every non-polymer residue |
| `protein` | every amino-acid residue |
| `interface` | (`scan` only) residues near the opposite partner |
| `*` | everything |

**Typical setups**

| System | `partner_a` | `partner_b` | `scan` |
|---|---|---|---|
| peptide–protein | `A` (protein) | `B` (peptide) | `B` |
| protein–protein | `A` | `B` | `interface` |
| protein–ligand | `protein` | `resname:XXX` | `interface` |

The two partners must not overlap. Only amino acids can be mutated, so
non-protein residues inside `scan` are skipped with a warning.

In [ ]:
#@title 7. Configure the scan { display-mode: "form" }
#@markdown ### Binding partners
partner_a = "A"  #@param {type:"string"}
partner_b = "B"  #@param {type:"string"}

#@markdown ### Positions to mutate
#@markdown A preset, or `custom` to use the `scan_custom` box below.
scan_preset = "quick demo (3 positions)"  #@param ["quick demo (3 positions)", "all of partner B", "interface residues", "custom"]
scan_custom = "B:19,23,26"  #@param {type:"string"}

#@markdown ### Substitutions
#@markdown `all` is the 20 canonical amino acids. Or give one-letter codes
#@markdown (`AVLIF`) or 3-letter codes (`ALA,TRP`).
mutations = "all"  #@param {type:"string"}

#@markdown ### Ensemble
#@markdown More conformers means better error bars and a longer run. The error
#@markdown bars come entirely from this number.
n_conformers = 8  #@param {type:"slider", min:2, max:40, step:1}
#@markdown FastRelax ramps per conformer.
relax_repeats = 2  #@param {type:"slider", min:1, max:5, step:1}
#@markdown Coordinate noise (Å) before relaxing, to widen the ensemble. Leave at
#@markdown 0 for a tight, fully relaxed ensemble; raise `relax_repeats` if you
#@markdown increase this.
perturb = 0.0  #@param {type:"slider", min:0.0, max:0.5, step:0.05}

#@markdown ### Protocol
#@markdown Minimise the mutated residue and its surroundings after packing.
#@markdown **Keep this on.** Without it, a large substitution into a tight
#@markdown pocket reports a clash artefact of tens or hundreds of REU instead
#@markdown of a usable ΔΔG. Costs roughly 3x the scan time.
minimize = True  #@param {type:"boolean"}
#@markdown Also record which energy term drives each ΔΔG.
decompose = True  #@param {type:"boolean"}
#@markdown Random seed, for reproducibility.
seed = 42  #@param {type:"integer"}
#@markdown FDR threshold for calling a substitution significant.
alpha = 0.05  #@param {type:"number"}

# ---------------------------------------------------------------------------
PRESETS = {
    "quick demo (3 positions)": scan_custom,
    "all of partner B": partner_b,
    "interface residues": "interface",
    "custom": scan_custom,
}
scan_selection = PRESETS[scan_preset].strip()
OUTDIR = WORKDIR / "results"

import sys

command = [
    sys.executable, "-m", "tmol_mutscan", "run",
    "-i", str(STRUCTURE),
    "--partner-a", partner_a,
    "--partner-b", partner_b,
    "--scan", scan_selection,
    "--mutations", mutations,
    "--n-conformers", str(n_conformers),
    "--relax-repeats", str(relax_repeats),
    "--perturb", str(perturb),
    "--seed", str(seed),
    "--alpha", str(alpha),
    "--device", DEVICE,
    "-o", str(OUTDIR),
]
if not minimize:
    command.append("--no-minimize")
if decompose:
    command.append("--decompose")

def _format_command(argv):
    """Echo the command with each flag on its own line, beside its value."""
    parts, group = [], []
    for token in argv:
        if token.startswith("-") and group:
            parts.append(" ".join(group))
            group = [token]
        else:
            group.append(token)
    if group:
        parts.append(" ".join(group))
    return " \\\n    ".join(parts)

print("command to be run:\n")
print("  " + _format_command(["tmol-mutscan", *command[3:]]))

# ---- work out the size of the job before committing to it -----------------
import torch as _torch
from tmol_mutscan.scan import parse_mutations
from tmol_mutscan.structure import load_structure, parse_selection, residue_table

print("\nresolving selections ...")
_loaded = load_structure(str(STRUCTURE), _torch.device("cpu"))
_pose = _loaded.pose
_a = parse_selection(_pose, partner_a, what="partner_a")
_b = parse_selection(_pose, partner_b, what="partner_b")
if int((_a & _b).sum()):
    raise SystemExit(
        "partner_a and partner_b overlap; make them disjoint before running"
    )

if "interface" in scan_selection.lower():
    _sa = parse_selection(_pose, scan_selection, interface_against=_b, what="scan")
    _sb = parse_selection(_pose, scan_selection, interface_against=_a, what="scan")
    _scan_mask = (_sa & _a) | (_sb & _b)
else:
    _scan_mask = parse_selection(_pose, scan_selection, what="scan")

_table = {r.block: r for r in residue_table(_pose)}
_positions = [
    _table[b]
    for b in _torch.nonzero(_scan_mask[0]).flatten().tolist()
    if _table[b].is_protein and bool(_a[0, b] or _b[0, b])
]
_subs = parse_mutations(mutations)
N_POSES = n_conformers * len(_positions) * (len(_subs) + 1)

print(f"  partner A        : {int(_a[0].sum())} residues")
print(f"  partner B        : {int(_b[0].sum())} residues")
print(f"  positions to scan: {len(_positions)}"
      f"  ({', '.join(f'{p.one_letter}{p.resnum}' for p in _positions[:25])}"
      f"{' ...' if len(_positions) > 25 else ''})")
print(f"  substitutions    : {len(_subs)}")
print(f"  ensemble         : {n_conformers} conformers")
print()
print(f"  TOTAL POSES TO BUILD: {N_POSES:,}")
print()
if DEVICE == "cpu":
    print("  On the CPU, budget very roughly 1 s per pose with minimisation on,")
    print(f"  so about {N_POSES / 3600:.1f} h. Reduce the pose count, or get a GPU.")
else:
    print("  Speed depends on the GPU Colab gave you, so the scan measures and")
    print("  prints its own poses/s and ETA as it goes. If the ETA that appears")
    print("  in cell 8 is too long, stop it and lower n_conformers, narrow the")
    print("  scan selection, or cut the substitution list.")
print()
print("  Remember free Colab disconnects after ~90 min idle and 12 h total.")

## 8. Run the scan

Progress streams below. Each chunk reports throughput in poses/second and an
ETA — **watch the first ETA**: if it is longer than you want to wait, interrupt
the cell (Runtime → Interrupt execution), lower `n_conformers` or narrow the
`scan` selection in cell 7, and run again.

Keep the browser tab alive. Colab stops a free session that looks idle.

In [ ]:
#@title 8. Run { display-mode: "form" }
import subprocess, time, sys

print("starting; this streams the live log\n")
started = time.perf_counter()

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    # Open Babel chatters on stderr during ligand preparation.
    if line.startswith(("=====", "*** Open Babel", "  No 2D or 3D")):
        continue
    sys.stdout.write(line)
    sys.stdout.flush()
process.wait()

elapsed = time.perf_counter() - started
print(f"\n{'=' * 72}")
print(f"finished in {elapsed / 60:.1f} min, exit code {process.returncode}")
if process.returncode == 0:
    # End-to-end, so it includes loading and ensemble relaxation as well as the
    # scan. The per-chunk "poses/s" in the log above is the scan rate alone,
    # which is the number to use when sizing a bigger run.
    print(f"overall: {N_POSES / elapsed:.2f} poses/s on {DEVICE} "
          "(including ensemble generation)")
    print("for sizing a larger run, use the per-chunk poses/s in the log above")
    print(f"results in: {OUTDIR}")
else:
    print("THE RUN FAILED - read the error above. Common causes:")
    print("  - a selection matched nothing: re-check cell 5's chain labels")
    print("  - CUDA out of memory: lower n_conformers, or add --batch-size 16")
print("=" * 72)

## 9. Figures

All figures are also saved as PDF alongside the PNGs shown here.

In [ ]:
#@title 9. Show the figures { display-mode: "form" }
from pathlib import Path
from IPython.display import Image, Markdown, display

CAPTIONS = {
    "ddg_mean_heatmap": (
        "**Mean ΔΔG matrix** — the headline result. Blue is predicted to bind "
        "*better*, red worse. The colour scale is clipped at the 98th "
        "percentile so one clash cannot flatten it. Cells on the wild-type "
        "diagonal are zero by construction."
    ),
    "ddg_sd_heatmap": (
        "**Standard deviation across the ensemble** — bright cells are "
        "substitutions whose effect depends on which conformer you ask, so "
        "treat their mean with caution."
    ),
    "ddg_significant_heatmap": (
        "**Significant substitutions only** — mean ΔΔG with everything failing "
        "the FDR threshold greyed out. This is the figure to trust."
    ),
    "top_mutations": (
        "**Ranked hits with 95% confidence intervals.** A bar whose interval "
        "crosses zero is not resolved. Solid bars passed FDR correction."
    ),
    "position_profile": (
        "**Tolerance per position** — the spread of every substitution in every "
        "conformer. A high, tight distribution marks a hotspot that resists "
        "change."
    ),
    "position_hotspots": (
        "**Hotspots vs tolerance.** Left = the wild-type residue already "
        "contributes much of the binding energy. High = the position resists "
        "substitution. Upper-left is a true hotspot (leave it alone); "
        "lower-right contributes little and tolerates change, which is where "
        "an improvement is most likely."
    ),
    "convergence": (
        "**Was the ensemble big enough?** Flat curves mean yes. Curves still "
        "falling at the right-hand edge mean you should raise `n_conformers`."
    ),
    "term_decomposition": (
        "**Which energy term drives each hit** — e.g. a large negative "
        "`fa_ljatr` means the substitution gains van der Waals contact, while "
        "`fa_lk` reflects desolvation and `hbond` a hydrogen bond gained or "
        "lost."
    ),
}

plots = Path(OUTDIR) / "plots"
shown = 0
for stem, caption in CAPTIONS.items():
    image = plots / f"{stem}.png"
    if image.is_file():
        display(Markdown(f"### {stem}\n{caption}"))
        display(Image(filename=str(image)))
        shown += 1

if shown == 0:
    print(f"no figures found in {plots} - did cell 8 succeed?")
else:
    print(f"{shown} figures shown; PDFs are in {plots}")

## 10. Tables

The numbers behind the figures. `ddg_statistics.csv` is the full table — one row
per (position, substitution) with every statistic.

**How to read a hit.** A substitution worth testing experimentally has all of:
`ddg_mean` clearly negative, `ci95_high` *also* below zero, `q_value < 0.05`,
`frac_stabilizing` near 1.0 (the whole ensemble agrees on the sign), and
`|cohens_dz|` above about 1.

Note that `ddg_sd` measures spread across *the ensemble you generated* — it does
not include the error of the energy function, the backbone held fixed during
mutation, or your input model. Small error bars mean the conformers agree with
each other, not that the ΔΔG is accurate.

In [ ]:
#@title 10. Show the result tables { display-mode: "form" }
#@markdown How many of the best substitutions to list.
show_top = 20  #@param {type:"slider", min:5, max:60, step:5}

import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

stats_dir = Path(OUTDIR) / "stats"
stats = pd.read_csv(stats_dir / "ddg_statistics.csv")
summary = pd.read_csv(stats_dir / "position_summary.csv")

real = stats[~stats.is_wt_self]
display(Markdown(
    f"### Best {show_top} substitutions\n"
    f"out of {len(real)} tested, "
    f"{int(real.significant.sum())} significant at FDR < {alpha}, "
    f"{int((real.effect == 'stabilizing').sum())} of them stabilizing"
))
display(
    real.sort_values("ddg_mean")
    .head(show_top)[[
        "position", "wt_aa", "mut_aa", "ddg_mean", "ddg_sd", "ddg_sem",
        "ci95_low", "ci95_high", "ddg_boltzmann", "frac_stabilizing",
        "cohens_dz", "q_value", "effect",
    ]]
    .round(3)
    .reset_index(drop=True)
)

display(Markdown(
    "### Per position\n"
    "`mean_ddg_all_substitutions` high = the position resists change. "
    "`wt_interface_contribution` very negative = the wild-type residue is "
    "already doing much of the binding."
))
columns = [
    "position", "wt_aa", "mean_ddg_all_substitutions", "best_substitution",
    "best_ddg_mean", "n_significant_stabilizing", "stabilizing_substitutions",
]
if "wt_interface_contribution" in summary.columns:
    columns.insert(2, "wt_interface_contribution")
_positions = summary[columns].round(3).copy()
_positions["stabilizing_substitutions"] = (
    _positions.stabilizing_substitutions.fillna("").replace("", "-")
)
display(_positions)

ensemble_scores = Path(OUTDIR) / "ensemble" / "ensemble_scores.csv"
if ensemble_scores.is_file():
    members = pd.read_csv(ensemble_scores)
    spread = members.total_score.max() - members.total_score.min()
    display(Markdown(
        "### Ensemble quality\n"
        "A wide `total_score` spread means the conformers are not equivalently "
        "relaxed, and the ΔΔG error bars inherit that. If the spread looks "
        "large for your system, raise `relax_repeats`."
    ))
    print(members.round(2).to_string(index=False))
    print(f"\ntotal_score spread: {spread:.2f} REU over {len(members)} conformers")
    print(f"interface energy  : {members.interface_energy.mean():.2f} "
          f"+/- {members.interface_energy.std(ddof=1):.2f} REU")

## 11. Download the results

In [ ]:
#@title 11. Download or save to Drive { display-mode: "form" }
#@markdown Download a zip of everything to your computer.
download_zip = True  #@param {type:"boolean"}
#@markdown Also copy the results into your Google Drive (mounts it if needed).
#@markdown Recommended for long runs, so results survive a disconnect.
save_to_drive = False  #@param {type:"boolean"}
#@markdown Folder inside your Drive to copy into.
drive_folder = "MyDrive/tmol-mutscan"  #@param {type:"string"}

import shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d-%H%M%S")
label = f"mutscan_{Path(STRUCTURE).stem}_{stamp}"
archive = shutil.make_archive(f"/content/{label}", "zip", root_dir=str(OUTDIR))
size_mb = Path(archive).stat().st_size / 1024**2
print(f"archive: {archive}  ({size_mb:.1f} MB)")

if save_to_drive:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    target = Path("/content/drive") / drive_folder.strip().lstrip("/") / label
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUTDIR, target, dirs_exist_ok=True)
    shutil.copy(archive, target.parent / Path(archive).name)
    print(f"copied to Drive: {target}")

if download_zip:
    from google.colab import files

    files.download(archive)
    print("download started (allow pop-ups if your browser blocked it)")

print("\ncontents:")
for path in sorted(Path(OUTDIR).rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTDIR)}")

---

## Tips, limits and troubleshooting

### Sizing a run

```
poses = n_conformers x positions x (substitutions + 1)
```

Cell 7 prints this number before you commit. Ways to cut it:

- **`scan interface`** instead of a whole chain — a residue far from the partner
  has no interface energy to change, so scanning it is wasted work.
- **fewer substitutions** — `--mutations AVLIFMW` for a hydrophobic-only scan is
  7/20 of the work.
- **fewer conformers** — but this is what your error bars are made of, so cut it
  last. Check `convergence.png` to see whether you had enough.
- **`minimize` off** — about 3x faster, but expect large positive ΔΔG artefacts
  wherever a substitution clashes. Only for a rough first pass.

### Surviving a disconnect

Free Colab drops a session after roughly 90 minutes idle or 12 hours total, and
does not guarantee a GPU. For a long scan:

1. Tick **save_to_drive** in cell 11.
2. Or split the work: run cell 8 once to build the ensemble, then scan subsets
   of positions separately with `tmol-mutscan scan --ensemble results/ensemble`,
   which reuses the relaxed ensemble instead of rebuilding it.

### Common problems

| Symptom | Cause and fix |
|---|---|
| `tmol.has_cuda` is `False` but `torch.cuda.is_available()` is `True` | The CPU-only TMol wheel got installed. Re-run cell 2 with `cuda_build = "cu12"`, then restart the session. |
| `CUDA out of memory` | Add `--batch-size 16` to the command in cell 7, or lower `n_conformers`. Memory grows with the square of the system size. |
| A selection "selected no residues" | Chain labels or numbering differ from what you assumed. Re-read cell 5 — mmCIF files often use different chain labels from the PDB version of the same entry. |
| Ligand ΔΔG looks nonsensical | You used a `.pdb` file. Re-fetch as mmCIF so bond orders survive. |
| Everything is destabilizing | Normal for a tight, well-evolved interface. Look at `position_hotspots.png` for the tolerant positions, which is where improvements live. |
| ΔΔG of hundreds of REU | A clash. Make sure `minimize` is on. Large substitutions into tight pockets stay pessimistic even so, because the backbone is held fixed. |

### What this method does and does not do

- Rosetta Energy Units are **not** calibrated kcal/mol. Use them to rank
  substitutions, not to predict an affinity. Validate hits experimentally.
- The backbone is **fixed** during mutation; only side chains and nearby atoms
  move. Large substitutions in tight pockets are therefore pessimistic.
- Proline and glycine change backbone preferences that a side-chain packer
  cannot model — treat their ΔΔG with particular suspicion.
- Disulfide-bonded cysteines and metal-coordinating residues cannot be
  substituted; those are skipped with a warning naming the position.
- The ensemble is a set of local minima near your input, not a Boltzmann
  ensemble.
- **Your input structure matters more than any setting here.** A scan of a
  badly modelled interface produces confident-looking numbers about nothing.

### More

- Repository and full documentation:
  <https://github.com/pablo-arantes/tmol-mutscan>
- TMol, which does the science:
  <https://github.com/uw-ipd/tmol>
- Every run writes `run_manifest.json` recording the full command, all resolved
  parameters and library versions — keep it with your results, and cite TMol if
  the work is published.